# Edgar demo: symbolic discovery on a toy dataset

This notebook walks through a minimal end-to-end workflow with `Edgar`.
We generate a synthetic dataset with a known symbolic form, define simple
seed programs/parameter estimators, preview the prompts/diagnostics, and
show how to launch the engine and inspect the resulting census snapshots.

> **Note:** Running the full engine still requires valid Google GenAI API
> credentials (\`GOOGLE_API_KEY\`). The cell that actually executes the
> agent is wrapped in a try/except so the rest of the demo can be explored
> even without credentials.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
import inspect

from entities import Program
from hypothesis_engine import Edgar, load_program_snapshots
import utils, tuning_curves_project


In [ ]:
rng = np.random.default_rng(0)
theta = np.linspace(0, 2 * np.pi, 200)

def ground_truth(theta, amplitude=1.0, bias=0.3, freq=2.0, phase=0.0):
    return bias + amplitude * np.cos(freq * theta - phase)

n_units = 6
X = np.tile(theta, (n_units, 1))
signals = []
for _ in range(n_units):
    amp = 0.9 + 0.25 * rng.standard_normal()
    bias = 0.2 * rng.random()
    phase = rng.uniform(0, np.pi)
    clean = ground_truth(theta, amplitude=amp, bias=bias, phase=phase)
    noisy = clean + 0.1 * rng.standard_normal(theta.shape)
    signals.append(noisy)
Y = np.asarray(signals)
print(X.shape, Y.shape)


In [ ]:
plt.figure(figsize=(10, 4))
for i in range(min(3, Y.shape[0])):
    plt.plot(theta, Y[i], label=f'unit {i}')
plt.title('Toy responses (ground truth + noise)')
plt.xlabel('theta (rad)')
plt.ylabel('spike rate (a.u.)')
plt.legend()
plt.show()


In [ ]:
def seed_model_sine_numpy(theta, A=1.0, B=0.3, phase=0.0):
    theta = np.asarray(theta)
    return B + A * np.sin(theta - phase)

def seed_model_sine_jax(theta, A=1.0, B=0.3, phase=0.0):
    theta = jnp.asarray(theta)
    return B + A * jnp.sin(theta - phase)

def seed_model_cosine_numpy(theta, A=1.0, B=0.2, phase=0.0):
    theta = np.asarray(theta)
    return B + A * np.cos(theta - phase)

def seed_model_cosine_jax(theta, A=1.0, B=0.2, phase=0.0):
    theta = jnp.asarray(theta)
    return B + A * jnp.cos(theta - phase)

def parameter_estimator(theta, spikes):
    span = float(np.max(spikes) - np.min(spikes))
    bias = float(np.median(spikes))
    phase = float(theta[np.argmax(spikes)])
    return np.array([max(span, 1e-3), bias, phase])

seed_programs = []
seed_specs = [
    (seed_model_sine_numpy, seed_model_sine_jax, 'sine_seed'),
    (seed_model_cosine_numpy, seed_model_cosine_jax, 'cos_seed'),
]
for idx, (fn_np, fn_jax, tag) in enumerate(seed_specs):
    params = jnp.tile(jnp.array([1.0, 0.2, 0.0]), (X.shape[0], 1))
    program = Program(
        function_code_string=inspect.getsource(fn_np),
        function=fn_jax,
        parameter_estimator_code_string=inspect.getsource(parameter_estimator),
        parameter_estimator=parameter_estimator,
        generation=-1,
        birth_island=-1,
        batch_index=idx,
        train_loss=0.0,
        params=params,
        initial_loss=0.0,
        initial_params=params[:1],
        llm_name=tag,
    )
    seed_programs.append(program)

seed_functions_numpy = [spec[0] for spec in seed_specs]
seed_functions_jax = [spec[1] for spec in seed_specs]
seed_parameter_estimators = [parameter_estimator for _ in seed_specs]


In [ ]:
prompt = utils.create_program_prompt(seed_programs, mode='explore', use_image=False, function_name='toy_model')
print('
'.join(prompt.splitlines()[:20]))


In [ ]:
tuning_curves_project.plot_model_fits(
    programs=[{"function": prog.function, "params": prog.params} for prog in seed_programs],
    loss_function=lambda pred, target: (pred - target) ** 2,
    x=jnp.asarray(X),
    y=jnp.asarray(Y),
    unit_selection=np.arange(4),
    labels=['seed 1', 'seed 2'],
    title='Seed fits (diagnostic example)',
)


In [ ]:
agent = Edgar(
    n_generations=1,
    n_islands=1,
    batch_size=1,
    use_image_feedback=False,
    time_limit=0.1,
    seed_functions_numpy=seed_functions_numpy,
    seed_functions_jax=seed_functions_jax,
    seed_parameter_estimators=seed_parameter_estimators,
    tiny_lm_name='gemini-1.5-flash',
    little_lm_name='gemini-2.0-flash',
    large_lm_name='gemini-2.5-flash',
)
try:
    agent.run(jnp.asarray(X), jnp.asarray(Y))
    print('Run finished, snapshots:', len(agent.snapshots_))
except Exception as exc:
    print('Edgar run skipped (likely missing API key):', exc)


In [ ]:
if agent.snapshots_:
    agent.plot_progress(metric='train')
    plt.show()
    agent.plot_lineage()
    plt.show()
else:
    print('No snapshots are available yet. Once a run completes you can use agent.plot_progress() and agent.plot_lineage().')


In [ ]:
if agent.snapshots_:
    for snap in agent.snapshots_[:3]:
        display(snap.to_dict())
else:
    print('Snapshots can also be reloaded later:')
    print('examples = load_program_snapshots('path/to/census.csv')')
